# Aula 07 — LLMOps: a jornada de um agente em produção (Databricks)

Notebook interativo da Aula 07. Construímos um **agente de suporte** sobre os primitivos nativos do
**Databricks Agent Framework** e percorremos o ciclo de LLMOps:

`SETUP → DESENVOLVER → VERSIONAR → FEEDBACK → GATE → SERVIR`

Cobrimos: **tools como UC Functions**, **RAG por Vector Search**, **observabilidade com autolog**,
**feedback humano**, **prompt registry**, **gate de aprovação (testes)** e **deploy em Model Serving**.

> Roda **dentro de um workspace Databricks** (precisa da sessão `spark` no setup e de credenciais).
> Os **guardrails** e os **painéis de monitoramento** ficam na UI do workspace.

## Ambiente
Instala as bibliotecas atuais e reinicia o Python (no Databricks).

In [ ]:
%pip install -U mlflow databricks-langchain databricks-vectorsearch unitycatalog-ai databricks-agents pytest

In [ ]:
try:
    dbutils.library.restartPython()  # noqa: F821
except NameError:
    pass

In [ ]:
import os, sys
try:
    _dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _dir = os.getcwd()
while _dir != os.path.dirname(_dir):
    if os.path.isdir(os.path.join(_dir, 'agente')):
        BASE = _dir; sys.path.insert(0, _dir); break
    _dir = os.path.dirname(_dir)

from agente.config import ENDPOINT_LLM, PROMPT_UC, MODELO_UC, INDICE_POLITICAS, LOJA
from agente.mlflow_databricks import credenciais_ok, configurar_mlflow

assert credenciais_ok(), 'Configure DATABRICKS_HOST/TOKEN no aula-07/.env (veja o .env.example).'
configurar_mlflow()
print('endpoint:', ENDPOINT_LLM, '| índice:', INDICE_POLITICAS, '| prompt:', f'{PROMPT_UC}@champion')

## Passo 0 — Provisionar os recursos no Unity Catalog

Idempotente: cria a tabela de **pedidos** (os dados reais que a função SQL de status consulta),
registra as **UC Functions** (tools), cria a tabela de políticas + o **índice de
Vector Search** (RAG) e registra a **versão produtiva do prompt** no Prompt Registry (alias
`champion`). A criação do índice leva alguns minutos na primeira vez.

In [ ]:
from agente.config import CATALOGO, SCHEMA
from agente.ferramentas import PEDIDOS, criar_tabela_pedidos, registrar_uc_functions
from agente.base_vetorial import criar_tabela_politicas, criar_indice, POLITICAS

# Tudo mora em aula_07.default — criamos o catálogo/schema (idempotente) antes de qualquer coisa.
spark.sql(f'CREATE CATALOG IF NOT EXISTS {CATALOGO}')          # noqa: F821 (spark é global)
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCHEMA}')  # noqa: F821
print('catálogo/schema:', f'{CATALOGO}.{SCHEMA}')

# A tabela de pedidos vem antes das funções: a função SQL de status referencia essa tabela.
print('tabela de pedidos:', criar_tabela_pedidos(spark), f'({len(PEDIDOS)} pedidos)')  # noqa: F821
print('UC Functions:', registrar_uc_functions())
tabela = criar_tabela_politicas(spark)  # noqa: F821 (spark é global no Databricks)
print('tabela de políticas:', tabela, f'({len(POLITICAS)} chunks)')
print('índice Vector Search:', criar_indice())

# O system prompt também é um recurso governado: 1ª versão produtiva -> alias champion.
from agente.prompt_registry import garantir_prompt_producao
print('prompt de produção:', f'{PROMPT_UC}@champion -> v{garantir_prompt_producao()}')

## Passo 1 — Desenvolver o agente (tools + RAG + observabilidade)

O `responder()` roda o agente (ChatDatabricks + UC Functions + retriever de Vector Search) com
`mlflow.langchain.autolog()` ligado: cada requisição vira uma **trace**. O system prompt vem do
**Prompt Registry** (alias `champion`) — cada resposta mostra a versão do prompt usada.

In [ ]:
from agente.agente_suporte import responder, TELEMETRIA

roteiro = [
    'Qual o status do pedido 1002?',                 # UC Function
    'Quanto fica o frete para o CEP 01000-000, 2 kg?',  # UC Function
    'Posso trocar um produto comprado há 20 dias?',  # RAG (Vector Search)
    'Qual a garantia dos eletrônicos?',              # RAG (Vector Search)
]
for p in roteiro:
    r = responder(p)
    print(f'\nQ: {p}\nA: {r.texto}\n   ferramentas: {r.ferramentas_usadas} | {r.latencia_ms:.0f} ms'
          f' | prompt v{r.prompt_versao} | trace {r.trace_id}')

### Observabilidade
Cada requisição acima virou uma **trace** (aba *Traces* no MLflow). Abaixo, um resumo da sessão.

In [ ]:
tools = [f for t in TELEMETRIA for f in t.ferramentas_usadas]
print('requisições :', len(TELEMETRIA))
print('chamadas tool:', len(tools), sorted(set(tools)))
print('>> abra a aba Traces no MLflow para inspecionar os spans.')

## Passo 2 — Versionar o prompt (MLflow Prompt Registry)

O prompt é o artefato que mais muda. Partimos da versão em produção (`champion`), registramos
uma **candidata** imutável (`challenger`), comparamos as duas e demonstramos promoção e rollback
— terminando com a produção intacta. No fluxo real, quem promove é o pipeline, após o gate.

In [ ]:
import difflib, re, mlflow
from agente.config import SYSTEM_PROMPT_PADRAO
from agente.prompt_registry import mover_alias, registrar_versao

def champion():   # cache_ttl_seconds=0: lê o alias na hora (o padrão faz cache de 60 s)
    return mlflow.genai.load_prompt(f'prompts:/{PROMPT_UC}@champion', cache_ttl_seconds=0)

producao = champion()
variante = SYSTEM_PROMPT_PADRAO + ' Ao final, pergunte se o cliente precisa de mais alguma ajuda.'
candidata = registrar_versao(variante, autor='Ana', mensagem='pergunta se pode ajudar em algo mais')
mover_alias('challenger', candidata)

frases = lambda t: re.split(r'(?<=[.!?])\s+', t.strip())   # uma frase por linha
print(*difflib.unified_diff(frases(producao.template), frases(variante), lineterm='', n=0,
      fromfile=f'v{producao.version} (champion)', tofile=f'v{candidata} (challenger)'), sep='\n')

mover_alias('champion', candidata)
print('\napós promover, champion -> v', champion().version)
mover_alias('champion', producao.version)   # rollback
print('após rollback, champion -> v', champion().version)

## Passo 3 — Feedback humano

O 👍/👎 do usuário é gravado com `mlflow.log_feedback` como *assessment* HUMAN ligada à trace.

In [ ]:
from agente.agente_suporte import registrar_feedback

r = responder('Qual a garantia dos eletrônicos?')
registrar_feedback(r, util=True, comentario='respondeu certo: 12 meses', usuario='cliente@acme.com')
print('👍 feedback gravado na trace', r.trace_id)

## Passo 4 — Gate de aprovação (testes unitários)

O gate é um `pytest` sobre situações conhecidas — as políticas da base e o status do pedido 1002
vindo dos dados reais da tabela `pedidos`. *Exit code* != 0 bloqueia o merge no CI. Rodamos o
mesmo arquivo do CI (a demo `98_demo_gate_cicd.py` mostra o gate falhando quando o registro some
da tabela).

In [ ]:
import sys
import pytest
# No /Workspace o pytest não consegue gravar __pycache__/.pytest_cache (Errno 95): desligamos
# essas escritas, senão a coleta falha antes de qualquer teste rodar.
sys.dont_write_bytecode = True
codigo = pytest.main([os.path.join(BASE, 'src', '04_gate_aprovacao.py'), '-v', '-ra',
                      '-p', 'no:cacheprovider',
                      # plugins (langsmith/anyio) já importados pelo notebook: aviso inofensivo
                      '-W', 'ignore::pytest.PytestAssertRewriteWarning'])
print('\n>> gate', 'VERDE ✅' if codigo == 0 else f'VERMELHO ❌ (exit {int(codigo)})')

## Passo 5 — Servir o agente (deploy real)

`log_model` (com `resources`) no **experimento de produção** → registro no Unity Catalog →
`agents.deploy`. O endpoint roda como o **Service Principal** do `00b_service_principal.py`
(credenciais no secret scope) e lê o prompt `@champion` do registry a cada requisição. Ele devolve o `trace_id` em `custom_outputs`, e é com ele que gravamos
o feedback do usuário. Leva alguns minutos. Descomente para provisionar o endpoint.

In [ ]:
# from mlflow.models.resources import (DatabricksServingEndpoint,
#     DatabricksVectorSearchIndex, DatabricksFunction)
# from agente.config import (ENDPOINT_LLM, ENDPOINT_EMBEDDINGS, CATALOGO, SCHEMA,
#     FUNC_STATUS_PEDIDO, FUNC_CALCULAR_FRETE)
# from agente.mlflow_databricks import usar_experimento_producao
# exp = usar_experimento_producao()   # modelo + traces/feedback do endpoint, separados das demos
# input_example = {'input': [{'role': 'user', 'content': 'Qual a garantia dos eletrônicos?'}]}
# with mlflow.start_run():
#     info = mlflow.pyfunc.log_model(
#         name='agente_suporte', python_model=os.path.join(BASE, 'agente', 'serving_model.py'),
#         code_paths=[os.path.join(BASE, 'agente')], input_example=input_example,
#         resources=[DatabricksServingEndpoint(endpoint_name=ENDPOINT_LLM),
#                    DatabricksServingEndpoint(endpoint_name=ENDPOINT_EMBEDDINGS),
#                    DatabricksVectorSearchIndex(index_name=INDICE_POLITICAS),
#                    DatabricksFunction(function_name=FUNC_STATUS_PEDIDO),
#                    DatabricksFunction(function_name=FUNC_CALCULAR_FRETE)],
#         registered_model_name=MODELO_UC,
#         pip_requirements=['mlflow','databricks-langchain','databricks-vectorsearch','unitycatalog-ai'])
# from databricks import agents
# from databricks.sdk import WorkspaceClient
# try:   # Free Edition: remove o deploy anterior (limite de concorrência provisionada)
#     agents.delete_deployment(MODELO_UC)
# except ValueError:
#     pass
# # environment_vars: endpoints/catálogo (lá não há .env) + a identidade do SP (Passo 0b)
# dep = agents.deploy(MODELO_UC, info.registered_model_version, scale_to_zero=True,
#                     environment_vars={'ENDPOINT_LLM_AGENTE': ENDPOINT_LLM,
#                                       'ENDPOINT_EMBEDDINGS': ENDPOINT_EMBEDDINGS,
#                                       'UC_CATALOG': CATALOGO, 'UC_SCHEMA': SCHEMA,
#                                       'MLFLOW_EXPERIMENT_ID': exp.experiment_id,
#                                       'DATABRICKS_HOST': WorkspaceClient().config.host,
#                                       'DATABRICKS_CLIENT_ID': '{{secrets/aula_07/client_id}}',
#                                       'DATABRICKS_CLIENT_SECRET': '{{secrets/aula_07/client_secret}}'})
# print('endpoint:', dep.endpoint_name, '— deploy iniciado (leva até ~15 min)')
print('Passo 5: veja aula-07/src/05_serving_deploy.py para o deploy completo.')

In [ ]:
# O agents.deploy só INICIA o deploy: consultar antes de terminar dá 404 "not ready to serve
# traffic yet". Esperamos o endpoint ficar READY; a resposta traz o trace_id -> o 👍 vai para a
# trace de produção.
# from datetime import timedelta
# from mlflow.deployments import get_deploy_client
# WorkspaceClient().serving_endpoints.wait_get_serving_endpoint_not_updating(
#     dep.endpoint_name, timeout=timedelta(minutes=30))
# resp = get_deploy_client('databricks').predict(endpoint=dep.endpoint_name, inputs=input_example)
# print(resp['output'][-1]['content'][0]['text'], '|', resp['custom_outputs'])
# registrar_feedback(resp['custom_outputs']['trace_id'], util=True, comentario='via endpoint')
print('Passo 5: espere o endpoint ficar READY antes de consultá-lo.')

## Conclusão

Percorremos o ciclo de LLMOps com governança: recursos no **Unity Catalog** (UC Functions + Vector
Search), **observabilidade** por autolog, **prompt registry**, **feedback humano**, **gate de
aprovação** por testes e **deploy** em Model Serving — a mesma esteira que o **GitHub Actions**
(`ci/ci-cd-agente.yml`) automatiza.